# Week 5 Exercises: Web Scraping & File Processing

**Data Engineering Course · Pipeline Africa**

---

Work through these in order — each section builds on the last.

**Setup:**
```
pip install requests beautifulsoup4 lxml pandas pyarrow openpyxl playwright psycopg2-binary
playwright install chromium
```

**Sections**

| Section | Exercises | Needs |
|---------|-----------|-------|
| 1 — HTML & CSS selectors | 1–6 | nothing (local files) |
| 2 — Static scraping | 7–12 | internet |
| 3 — Playwright | 13–18 | internet + chromium |
| 4 — File formats | 19–22 | nothing |
| 5 — A small pipeline | 23–24 | internet + PostgreSQL |

Most exercises tell you the answer you should get. If your number does not match, your selector is wrong — do not move on.

**Be polite:** every exercise that hits the internet must include `time.sleep(1)` between requests. The sandboxes we use do not mind, but the habit is what you are building.

In [ ]:
import re
import time
import json
import requests
import pandas as pd
from io import StringIO
from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin

DATA  = Path('../data')
PAGES = DATA / 'pages'
CLEAN = DATA / 'clean'
OUT   = DATA / 'exercises'
for p in [CLEAN, OUT]:
    p.mkdir(parents=True, exist_ok=True)

print('Ready. Practice pages:', [p.name for p in sorted(PAGES.glob('*.html'))])

---

## Section 1: HTML & CSS Selectors (Exercises 1–6)

These use the saved pages in `../data/pages/`. No internet needed — open `job_board.html` in a text editor before you start.

### Exercise 1

Read `../data/pages/job_board.html` and parse it with BeautifulSoup.

Print:
- the page's `<title>` text
- how many `article.job-card` elements it contains
- the text of the `h1.site-title`

*You should find 6 job cards.*

In [ ]:
# Exercise 1 — your code here

### Exercise 2

Loop over every job card and print one line per job:

```
Data Engineer            Paystack             Lagos, Nigeria
```

Use `select_one` with CSS selectors, and `get_text(strip=True)` — one of the titles has stray whitespace around it in the HTML.

In [ ]:
# Exercise 2 — your code here

### Exercise 3

The posted date appears twice in each card: as human text (`2 March 2026`) and as a machine-readable attribute (`datetime="2026-03-02"`).

Extract the **attribute** version for all six cards, convert it with `pd.to_datetime`, and print the job title of the **most recently posted** job.

*Hint: the attribute is on the `<time>` element inside `.posted`.*

In [ ]:
# Exercise 3 — your code here

### Exercise 4

Not every card has a `.salary` element.

1. Print how many cards **do** have one (use a selector that finds `.salary` elements).
2. Print the `data-job-id` of every card that does **not** have one.
3. Explain in a comment why `card.select_one('.salary').get_text()` is dangerous here.

*Two cards are missing a salary.*

In [ ]:
# Exercise 4 — your code here

### Exercise 5

Write a helper function:

```python
def text_at(node, selector, default=None):
    ...
```

It returns the stripped text of the first match for `selector` inside `node`, or `default` when there is no match.

Then use it to build a DataFrame of all six jobs with these columns:

`job_id`, `title`, `company`, `location`, `job_type`, `salary`, `posted`, `n_tags`

- `job_id` must be an integer
- `posted` must be a datetime
- `salary` must be `None` where the element is missing
- `n_tags` is how many `.tag` elements the card has

Print `df.dtypes` and `df.isna().sum()`.

In [ ]:
# Exercise 5 — your code here

### Exercise 6

`job_board.html` is page 1 of 2. Its pager contains `<a class="next" href="job_board_page2.html">`.

Write a loop that:
1. starts at `job_board.html`
2. parses the cards on the current page
3. reads the `.next` link's `href` and follows it
4. stops when there is no `.next` link

Combine everything into one DataFrame with an extra `source_page` column holding the filename each row came from.

*You should end with 10 rows from 2 pages.*

In [ ]:
# Exercise 6 — your code here

---

## Section 2: Static Scraping (Exercises 7–12)

Live requests against **books.toscrape.com**, a sandbox built for this. Sleep one second between requests.

### Exercise 7

Write a `fetch(url)` function that:
- uses a `requests.Session` with a `User-Agent` header identifying your scraper
- passes `timeout=15`
- calls `raise_for_status()`
- sets `response.encoding = 'utf-8'`
- returns a BeautifulSoup tree

Use it to print the `<title>` of `https://books.toscrape.com/`.

Then call it on `https://books.toscrape.com/no-such-page.html` inside a `try/except requests.HTTPError` and print the error.

In [ ]:
# Exercise 7 — your code here

### Exercise 8

Fetch the homepage **twice**: once without setting the encoding, once with `response.encoding = 'utf-8'`.

Print the first book's `.price_color` text from each. Explain the difference in a comment.

*The broken one shows `Â£51.77`.*

In [ ]:
# Exercise 8 — your code here

### Exercise 9

Scrape every book on page 1 of books.toscrape.com into a DataFrame with:

| Column | Where it comes from |
|--------|---------------------|
| `title` | the `title` **attribute** of the link in `h3` (not its text — that is truncated) |
| `price_gbp` | `.price_color`, as a **float** with no currency symbol |
| `rating` | the second class on `p.star-rating` (`One`…`Five`), mapped to 1–5 |
| `in_stock` | `True` if `.availability` contains "in stock" |
| `url` | the absolute URL, built with `urljoin` |

Print the shape, the dtypes, and the 3 most expensive books.

*20 books. `price_gbp` must be a float, not an object.*

In [ ]:
# Exercise 9 — your code here

### Exercise 10

Turn Exercise 9 into a paginated scrape.

Write `scrape_pages(start_url, max_pages)` that follows the `li.next a` link, sleeps 1 second between pages, prints progress, and returns one combined DataFrame.

Run it for **3 pages**. Then print:
- the total row count
- the most expensive book across all three pages
- the mean price per rating value (`groupby('rating')`)

*60 rows.*

In [ ]:
# Exercise 10 — your code here

### Exercise 11

Take the URL of the first book from Exercise 10 and scrape its **detail page** for:

- `category` (from the breadcrumb — the last link in `ul.breadcrumb`)
- `upc`, from the product information table
- `stock_count` — the number inside `In stock (22 available)`, as an integer
- `description` — the paragraph that follows the `#product_description` heading

Print all four. *Hint: the description is a sibling of the heading, not a child — `find_next_sibling('p')`.*

In [ ]:
# Exercise 11 — your code here

### Exercise 12

`../data/pages/salaries_table.html` contains two HTML tables.

1. Read **both** with one `pd.read_html()` call and print each one's shape.
2. Read **only** the open-roles table using the `match=` argument.
3. From the salary benchmarks table, add a column `senior_multiple` = `Senior / Junior`, rounded to 1 decimal, and print the roles sorted by it.

*Which roles gain the most from junior to senior — and is the answer as clear-cut as you expected? Say what you would tell a stakeholder who asked for "the" top role.*

In [ ]:
# Exercise 12 — your code here

---

## Section 3: Playwright (Exercises 13–18)

Remember: in a notebook you need the **async** API, and every Playwright call needs `await`.

```python
from playwright.async_api import async_playwright
pw = await async_playwright().start()
browser = await pw.chromium.launch()
```

Close the browser at the end of the section (Exercise 18), not after each exercise.

### Exercise 13

Start Playwright, launch headless chromium, open a page, and go to `https://quotes.toscrape.com/js/`.

Print:
- the page title
- how many `.quote` elements the page has, via a locator
- how many `.quote` elements `requests` + BeautifulSoup find on the **same URL**

*10 and 0. Write a one-line comment explaining why they differ.*

In [ ]:
# Exercise 13 — your code here

### Exercise 14

Using **locators only** (no BeautifulSoup), build a DataFrame of the 10 quotes on that page with columns `text`, `author`, `tags`.

- strip the curly quote marks `“ ”` off the text
- `tags` should be a comma-separated string

*Hint: `await loc.all()` gives you one locator per match; `all_inner_texts()` gets every tag at once.*

In [ ]:
# Exercise 14 — your code here

### Exercise 15

Do the same extraction with the **hybrid pattern**: get `await page.content()` and parse it with BeautifulSoup.

Write it as a function `parse_quotes(html)` that takes an HTML string and returns a list of dicts — no `page`, no `await` inside it.

Assert that it returns the same number of rows as Exercise 14, and explain in a comment why a function with this signature is easier to test.

In [ ]:
# Exercise 15 — your code here

### Exercise 16

Click through the first **3 pages** of `https://quotes.toscrape.com/js/`, collecting quotes with `parse_quotes()`.

Your loop must:
- `wait_for_selector('.quote')` on each page before extracting
- find `li.next a` and stop if it is not there
- not click Next on the final wanted page
- print the URL and row count for each page

*30 rows, 10 per page.*

In [ ]:
# Exercise 16 — your code here

### Exercise 17

`https://quotes.toscrape.com/scroll` loads more quotes as you scroll.

Scrape all of them:
1. go to the page and wait for `.quote`
2. loop: scroll to the bottom, then wait for the `.quote` count to **increase**
3. stop when it stops increasing, or after a cap of 15 rounds
4. print the count after each round

How many quotes are on that page in total? Why is "scroll 10 times" a worse stopping rule than the one you just wrote?

In [ ]:
# Exercise 17 — your code here

### Exercise 18

The scroll page gets its data from a hidden JSON API.

1. Attach `page.on('response', ...)` and reload the page. Print every response whose `resource_type` is `xhr` or `fetch`.
2. Identify the endpoint and its query parameter.
3. Using **`requests` only — no browser** — fetch every page of that API until `has_next` is `False`, and build a DataFrame of all the quotes.
4. Compare the row count with Exercise 17, and time both approaches with `time.perf_counter()`.
5. Close the browser: `await browser.close()` then `await pw.stop()`.

Write one sentence on what this means for how you should start any scraping task.

In [ ]:
# Exercise 18 — your code here

---

## Section 4: File Formats (Exercises 19–22)

No internet needed — use the DataFrames you already built, or re-read `../data/clean/books.csv` from Lesson 2.

### Exercise 19

Take your 60-book DataFrame from Exercise 10 (or read `../data/clean/books.csv`).

Save it four ways into `../data/exercises/`:
- `books.csv`
- `books.jsonl` (`orient='records', lines=True`)
- `books.parquet` (snappy)
- `books_gzip.parquet` (gzip)

Print a table of filename, size in KB, and size relative to the CSV. Which is smallest? Which is largest?

In [ ]:
# Exercise 19 — your code here

### Exercise 20

A dtype round-trip test.

1. Add a real date column to your books DataFrame: `df['scraped_at'] = pd.Timestamp.now()`
2. Write it to CSV and to Parquet
3. Read both back
4. Build a comparison DataFrame with three columns — `original`, `from_csv`, `from_parquet` — holding the dtypes of each

Which columns survive CSV? Which do not? Write the one-line rule this teaches you.

In [ ]:
# Exercise 20 — your code here

### Exercise 21

Load `../data/sample_jobs.json`.

1. Flatten `raw['jobs']` with `pd.json_normalize` and print the resulting column names.
2. Rename the dotted columns to flat ones (`company.name` → `company`, and so on).
3. Produce a **second** DataFrame with one row per `(job, tag)` using `record_path='tags'` and `meta=['job_id', 'title']`.
4. Print the three most requested tags.

*6 jobs become 16 job-tag rows.*

In [ ]:
# Exercise 21 — your code here

### Exercise 22

Build an Excel report at `../data/exercises/book_report.xlsx` with two sheets:

- `books` — the full 60-row table
- `by_rating` — count and mean price grouped by rating

Then read the workbook back with `sheet_name=None` and print each sheet's name and shape.

Finally, write one comment explaining why you would not use this file as the hand-off between two stages of a pipeline.

In [ ]:
# Exercise 22 — your code here

---

## Section 5: A Small Pipeline (Exercises 23–24)

Put it together. These two mirror Lab 5 at a smaller scale, with `requests` instead of Playwright.

### Exercise 23

Write a **script**, not cells.

Create `scrape_books.py` in this folder (use `%%writefile`) that:

- takes `--pages N` (default 2) and `--out PATH` (default `../data/exercises/books_raw.jsonl`) with `argparse`
- scrapes N pages of books.toscrape.com by following the `next` link
- sleeps 1 second between pages
- logs progress with the `logging` module to both the console and `../logs/exercises.log`
- writes **one JSON object per line** — raw strings, no cleaning, plus a `scraped_at` timestamp
- prints the row count at the end

Then run it from a cell: `!python scrape_books.py --pages 3`

*You should get 60 lines.*

In [ ]:
# Exercise 23 — write your script in a %%writefile cell, then run it

### Exercise 24

Load `books_raw.jsonl` and finish the pipeline:

1. **Clean**: price to float, rating word to integer, `in_stock` to boolean, strip all strings
2. **Validate**: assert no nulls in `title`/`price_gbp`/`url`, and that `url` is unique. Print the row count
3. **Save** `../data/clean/books.parquet`
4. **Load** into PostgreSQL:
   - create a database `books_db` (in pgAdmin or with `CREATE DATABASE books_db;`)
   - create a table `books (url TEXT PRIMARY KEY, title TEXT NOT NULL, price_gbp NUMERIC(8,2), rating INT, in_stock BOOLEAN, scraped_at TIMESTAMPTZ)`
   - upsert with `ON CONFLICT (url) DO UPDATE`
   - run the load **twice** and assert the row count does not change
5. **Answer in SQL**:
   - average price by rating, ordered by rating
   - the 5 most expensive books
   - how many books are out of stock

Read your database credentials from the environment — **do not type your password into the notebook.**

In [ ]:
# Exercise 24 — your code here

---

## When You Are Done

- [ ] Section 1 (Ex 1–6): all six complete, 10-row combined DataFrame in Ex 6
- [ ] Section 2 (Ex 7–12): all six complete, 60 books scraped in Ex 10
- [ ] Section 3 (Ex 13–18): all six complete, browser closed at the end of Ex 18
- [ ] Section 4 (Ex 19–22): all four complete, size and dtype comparisons printed
- [ ] Section 5 (Ex 23–24): `scrape_books.py` runs from the command line; `books` table loads idempotently
- [ ] No passwords anywhere in the notebook
- [ ] Notebook runs top to bottom with no errors
- [ ] Pushed to GitHub

**If you want more:** scrape all 50 pages of books.toscrape.com (1,000 books) with your Exercise 23 script, save it as partitioned Parquet with `partition_cols=['rating']` — remembering to sort first — and compare the total size against a single CSV of the same data.